# 4주차 B1 · UCI HAR Inception-ResNet 1D

UCI HAR의 128시점·9채널 센서 신호에 **Inception 1D와 residual connection을 함께 적용**합니다.

이번 실험에서 확인하려는 질문은 다음과 같습니다.

1. 서로 다른 길이의 Conv1D 필터가 짧고 긴 동작 패턴을 함께 찾을 수 있는가?
2. Inception 블록을 깊게 쌓을 때 residual `Add`가 학습을 안정시키는가?
3. 정확도뿐 아니라 파라미터 수와 실행 시간까지 보았을 때 효율적인가?

**실행 상태:** 이 파일은 Colab 실행 전 실험 설계본입니다. 저장된 점수를 실제 결과로 간주하지 않습니다.

**중요:** 모델 선택은 피험자 단위 validation으로 수행하고, 공식 test는 최종 구조를 정한 뒤 한 번만 확인합니다.

## 0. B1 구조와 비교 원칙

Inception 내부의 여러 필터 결과는 채널 방향으로 `Concatenate`합니다. 그다음 1×1 Conv1D로 채널 수를 맞춘 Inception 출력 $F(x)$와 입력 shortcut $x$를 `Add`합니다.

$$y=\operatorname{ReLU}(x+\alpha F(x))$$

```text
입력 x
 ├─ bottleneck 1×1 → Conv1D(3)
 ├─ bottleneck 1×1 → Conv1D(7)
 ├─ bottleneck 1×1 → Conv1D(15)
 └─ MaxPool(3)      → Conv1D(1)
             ↓ Concatenate
          projection 1×1 = F(x)
             ↓ residual scaling α
          Add([x, αF(x)])
             ↓ ReLU
```

- `Concatenate`: 서로 다른 시간 범위의 특징을 채널 방향으로 모읍니다.
- `Add`: Inception 출력과 shortcut을 더합니다. 두 텐서의 shape이 같아야 합니다.
- `projection`: 채널 수가 다르면 1×1 Conv1D로 shape을 맞춥니다.
- `residual_scale`: 넓은 residual branch가 학습 초기에 지나치게 커지는 것을 줄입니다.

이번 B1에는 SE, Depthwise convolution, Transformer를 넣지 않습니다. 여러 방법을 동시에 넣으면 무엇이 결과에 영향을 주었는지 구분하기 어렵기 때문입니다.

## 1. 라이브러리 불러오기

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

## 2. 실험 설정

`SEED`를 0, 1, 2로 바꾸어 세 번 실행합니다. `USE_RESIDUAL=False`는 나중에 shortcut 효과만 확인하기 위한 대조 실험입니다. 먼저 B1의 기본값인 `True`로 실행합니다.

In [ ]:
SEED = 0
USE_RESIDUAL = True
RESIDUAL_SCALE = 0.2

np.random.seed(SEED)
tf.random.set_seed(SEED)

EXPERIMENT_NAME = "B1-residual" if USE_RESIDUAL else "B1-no-skip"
print("experiment:", EXPERIMENT_NAME, "| seed:", SEED)

## 3. Google Drive와 데이터 경로 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"
assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"

## 4. 9개 센서 신호와 데이터 로더 정의

In [ ]:
signals = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]


def load_data(split):
    channels = []

    for signal in signals:
        path = f"{DATA_PATH}/{split}/Inertial Signals/{signal}_{split}.txt"
        channels.append(np.loadtxt(path))

    # (9, N, 128)이 아니라 각 (N, 128) 배열을 마지막 축에 쌓아 (N, 128, 9)로 만듭니다.
    X = np.stack(channels, axis=-1)
    y = np.loadtxt(f"{DATA_PATH}/{split}/y_{split}.txt").astype(int) - 1
    subject = np.loadtxt(f"{DATA_PATH}/{split}/subject_{split}.txt").astype(int)

    return X, y, subject


X_train_all, y_train_all, s_train_all = load_data("train")
X_test, y_test, s_test = load_data("test")

print("official train:", X_train_all.shape)
print("official test :", X_test.shape, "(최종 셀 전까지 평가하지 않음)")

## 5. 피험자 단위 validation 분리

같은 사람의 윈도우가 학습과 validation에 동시에 들어가지 않도록 피험자 5·11·17·23·29를 통째로 분리합니다. 특히 5번과 17번의 `SITTING`/`STANDING` 성능을 핵심 지표로 봅니다.

In [ ]:
VAL_SUBJECTS = [5, 11, 17, 23, 29]
FOCUS_SUBJECTS = [5, 17]

val_mask = np.isin(s_train_all, VAL_SUBJECTS)

X_val = X_train_all[val_mask]
y_val = y_train_all[val_mask]
s_val = s_train_all[val_mask]

X_train = X_train_all[~val_mask]
y_train = y_train_all[~val_mask]
s_train = s_train_all[~val_mask]

print("train:", X_train.shape, "subjects:", sorted(np.unique(s_train)))
print("val  :", X_val.shape, "subjects:", sorted(np.unique(s_val)))

## 6. 학습 데이터 통계로 정규화

In [ ]:
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = ((X_train - mean) / (std + 1e-8)).astype("float32")
X_val = ((X_val - mean) / (std + 1e-8)).astype("float32")
X_test = ((X_test - mean) / (std + 1e-8)).astype("float32")

n_outputs = 6
y_train_onehot = to_categorical(y_train, num_classes=n_outputs)
y_val_onehot = to_categorical(y_val, num_classes=n_outputs)

## 7. Inception-residual 1D 블록

세 Conv1D 경로는 kernel 3·7·15로 서로 다른 시간 범위를 봅니다. 먼저 1×1 bottleneck으로 채널을 줄여 큰 kernel의 계산량을 제한합니다. pooling 경로는 지역적으로 강한 값을 보존합니다.

`filters`는 4의 배수로 두어 네 경로의 출력 채널 합이 같게 합니다. 마지막 projection과 shortcut의 shape을 확인한 뒤 `Add`합니다.

In [ ]:
def inception_residual_block(
    x,
    filters,
    bottleneck_filters=16,
    kernel_sizes=(3, 7, 15),
    residual_scale=0.2,
    use_residual=True,
    name="ir",
):
    if filters % 4 != 0:
        raise ValueError("filters는 네 branch에 나눌 수 있도록 4의 배수여야 합니다.")

    branch_filters = filters // 4
    shortcut = x

    # 큰 kernel을 적용하기 전에 채널을 줄여 계산량을 제한합니다.
    bottleneck = layers.Conv1D(
        bottleneck_filters, 1, padding="same", use_bias=False,
        name=f"{name}_bottleneck",
    )(x)
    bottleneck = layers.BatchNormalization(name=f"{name}_bottleneck_bn")(bottleneck)
    bottleneck = layers.Activation("relu", name=f"{name}_bottleneck_relu")(bottleneck)

    branches = []
    for kernel_size in kernel_sizes:
        branch = layers.Conv1D(
            branch_filters, kernel_size, padding="same", use_bias=False,
            name=f"{name}_conv_k{kernel_size}",
        )(bottleneck)
        branches.append(branch)

    pool_branch = layers.MaxPooling1D(
        pool_size=3, strides=1, padding="same", name=f"{name}_pool"
    )(x)
    pool_branch = layers.Conv1D(
        branch_filters, 1, padding="same", use_bias=False,
        name=f"{name}_pool_projection",
    )(pool_branch)
    branches.append(pool_branch)

    merged = layers.Concatenate(axis=-1, name=f"{name}_concatenate")(branches)

    # Inception 결과를 shortcut과 같은 채널 수로 투영합니다.
    residual = layers.Conv1D(
        filters, 1, padding="same", use_bias=False,
        name=f"{name}_residual_projection",
    )(merged)
    residual = layers.BatchNormalization(name=f"{name}_residual_bn")(residual)

    if use_residual:
        if x.shape[-1] != filters:
            shortcut = layers.Conv1D(
                filters, 1, padding="same", use_bias=False,
                name=f"{name}_shortcut_projection",
            )(shortcut)
            shortcut = layers.BatchNormalization(name=f"{name}_shortcut_bn")(shortcut)

        residual = layers.Rescaling(
            residual_scale, name=f"{name}_residual_scale"
        )(residual)
        x = layers.Add(name=f"{name}_add")([shortcut, residual])
    else:
        # shortcut만 뺀 대조 실험입니다. 나머지 Inception 경로는 동일합니다.
        x = residual

    return layers.Activation("relu", name=f"{name}_out")(x)

## 8. B1 모델 만들기

32채널 블록 두 개와 64채널 블록 두 개를 사용합니다. 중간의 pooling으로 시간축을 128에서 64로 줄입니다. `GlobalAveragePooling1D`는 큰 Flatten/Dense head를 피하여 파라미터 수를 줄입니다.

In [ ]:
def build_b1_model(use_residual=True, residual_scale=0.2):
    inputs = layers.Input(shape=(128, 9), name="sensor_window")

    x = layers.Conv1D(32, 5, padding="same", use_bias=False, name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("relu", name="stem_relu")(x)

    x = inception_residual_block(
        x, filters=32, bottleneck_filters=16,
        residual_scale=residual_scale, use_residual=use_residual, name="ir32_1",
    )
    x = inception_residual_block(
        x, filters=32, bottleneck_filters=16,
        residual_scale=residual_scale, use_residual=use_residual, name="ir32_2",
    )

    x = layers.MaxPooling1D(2, name="downsample_pool")(x)
    x = layers.Conv1D(64, 1, padding="same", use_bias=False, name="channel_up")(x)
    x = layers.BatchNormalization(name="channel_up_bn")(x)
    x = layers.Activation("relu", name="channel_up_relu")(x)

    x = inception_residual_block(
        x, filters=64, bottleneck_filters=24,
        residual_scale=residual_scale, use_residual=use_residual, name="ir64_1",
    )
    x = inception_residual_block(
        x, filters=64, bottleneck_filters=24,
        residual_scale=residual_scale, use_residual=use_residual, name="ir64_2",
    )

    x = layers.GlobalAveragePooling1D(name="global_average_pooling")(x)
    x = layers.Dropout(0.4, name="classifier_dropout")(x)
    outputs = layers.Dense(n_outputs, activation="softmax", name="activity")(x)

    return Model(inputs, outputs, name="UCI_HAR_B1_Inception_ResNet1D")


model = build_b1_model(
    use_residual=USE_RESIDUAL,
    residual_scale=RESIDUAL_SCALE,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
print(f"parameters: {model.count_params():,}")

## 9. `Concatenate`와 `Add` shape 확인

`Concatenate` 뒤에는 여러 branch의 채널이 모이고, projection을 거친 residual과 shortcut은 같은 shape이어야 합니다. 이 검사는 잘못된 `Add` 설계를 실행 전에 발견하기 위한 것입니다.

In [ ]:
for block_name in ["ir32_1", "ir32_2", "ir64_1", "ir64_2"]:
    merged_shape = model.get_layer(f"{block_name}_concatenate").output.shape
    residual_shape = model.get_layer(f"{block_name}_residual_bn").output.shape
    output_shape = model.get_layer(f"{block_name}_out").output.shape
    print(block_name, "concat:", merged_shape, "residual:", residual_shape, "out:", output_shape)

    if USE_RESIDUAL:
        add_layer = model.get_layer(f"{block_name}_add")
        left_shape, right_shape = [tensor.shape for tensor in add_layer.input]
        assert left_shape == right_shape, f"{block_name} Add shape 불일치"

## 10. 학습

기존 S0와 마찬가지로 최대 100에폭, batch size 64, Adam과 EarlyStopping을 사용합니다. 구조 외의 조건을 가능한 한 고정하고 학습 시간도 함께 기록합니다.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=10,
        restore_best_weights=True, verbose=1,
    )
]

train_start = time.perf_counter()

history = model.fit(
    X_train,
    y_train_onehot,
    validation_data=(X_val, y_val_onehot),
    epochs=100,
    batch_size=64,
    shuffle=True,
    callbacks=callbacks,
    verbose=1,
)

training_seconds = time.perf_counter() - train_start
best_epoch = int(np.argmin(history.history["val_loss"])) + 1

print(f"training time: {training_seconds:.1f} sec")
print(f"best epoch by val_loss: {best_epoch}")

## 11. Validation 전체 평가

In [ ]:
class_names = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING",
]

y_val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
cm_val = confusion_matrix(y_val, y_val_pred, labels=range(n_outputs))

print(f"validation accuracy : {val_accuracy:.4f}")
print(f"validation macro F1 : {val_f1:.4f}")
print(classification_report(
    y_val, y_val_pred, target_names=class_names,
    digits=4, zero_division=0,
))
print("confusion matrix")
print(cm_val)

## 12. `SITTING`/`STANDING`과 피험자별 평가

전체 정확도가 좋아져도 기존 핵심 문제인 앉음↔섬 오류가 늘어날 수 있으므로 별도로 기록합니다.

In [ ]:
SIT, STAND = 3, 4

print("subject | all acc | SIT/STAND acc | SIT->STAND | STAND->SIT")
for subject in VAL_SUBJECTS:
    mask = s_val == subject
    ss_mask = mask & np.isin(y_val, [SIT, STAND])
    sit_to_stand = np.sum(mask & (y_val == SIT) & (y_val_pred == STAND))
    stand_to_sit = np.sum(mask & (y_val == STAND) & (y_val_pred == SIT))
    mark = " <- focus" if subject in FOCUS_SUBJECTS else ""

    print(
        f"{subject:7d} | {accuracy_score(y_val[mask], y_val_pred[mask]):.4f}  | "
        f"{accuracy_score(y_val[ss_mask], y_val_pred[ss_mask]):.4f}        | "
        f"{sit_to_stand:10d} | {stand_to_sit:10d}{mark}"
    )

focus_mask = np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])
other_mask = ~np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

focus_ss_acc = accuracy_score(y_val[focus_mask], y_val_pred[focus_mask])
other_ss_acc = accuracy_score(y_val[other_mask], y_val_pred[other_mask])
val_ss_err = int(cm_val[SIT, STAND] + cm_val[STAND, SIT])
val_dynamic_err = int(cm_val[0:3, 0:3].sum() - np.trace(cm_val[0:3, 0:3]))

## 13. 추론 시간 측정

GPU·CPU 종류에 따라 절대 시간은 달라집니다. 따라서 같은 Colab 런타임에서 다른 모델과 측정한 값끼리만 비교합니다.

In [ ]:
timing_batch = tf.convert_to_tensor(X_val[:256])

# 초기 그래프 준비와 장치 준비 시간을 제외하기 위한 warm-up입니다.
for _ in range(3):
    model(timing_batch, training=False).numpy()

repeats = 20
inference_start = time.perf_counter()
for _ in range(repeats):
    model(timing_batch, training=False).numpy()
inference_seconds = time.perf_counter() - inference_start

inference_ms_per_sample = (
    inference_seconds * 1000 / (repeats * len(timing_batch))
)
print(f"inference: {inference_ms_per_sample:.4f} ms/sample")

## 14. Gradient norm 관찰

고정된 validation mini-batch에서 loss를 한 번 계산하고 각 Conv1D kernel의 gradient norm을 출력합니다. 앞쪽 층의 gradient가 반복적으로 매우 작고 residual을 끈 실험보다도 작다면 gradient 전달 문제를 의심할 수 있습니다.

단, 한 시점의 gradient norm만으로 gradient 소실을 증명할 수는 없습니다. 동일 seed에서 `USE_RESIDUAL=True/False`를 비교하고 학습곡선·수렴 에폭·정확도를 함께 봅니다.

In [ ]:
probe_x = tf.convert_to_tensor(X_val[:64])
probe_y = tf.convert_to_tensor(y_val_onehot[:64])

with tf.GradientTape() as tape:
    probe_prediction = model(probe_x, training=False)
    probe_loss = tf.reduce_mean(
        tf.keras.losses.categorical_crossentropy(probe_y, probe_prediction)
    )

gradients = tape.gradient(probe_loss, model.trainable_variables)
gradient_rows = []

for variable, gradient in zip(model.trainable_variables, gradients):
    variable_path = getattr(variable, "path", variable.name)
    if gradient is not None and "kernel" in variable_path:
        gradient_rows.append((variable_path, float(tf.norm(gradient))))

print("loss:", float(probe_loss))
print("\n앞쪽 kernel gradient norm")
for name, norm in gradient_rows[:6]:
    print(f"{name:55s} {norm:.8f}")

print("\n뒤쪽 kernel gradient norm")
for name, norm in gradient_rows[-6:]:
    print(f"{name:55s} {norm:.8f}")

## 15. 학습곡선

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} loss · seed {SEED}")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} accuracy · seed {SEED}")
plt.legend()

plt.tight_layout()
plt.show()

## 16. 결과 한 줄 요약

In [ ]:
print(
    f"{EXPERIMENT_NAME} | seed {SEED} | "
    f"val acc {val_accuracy:.4f} | val F1 {val_f1:.4f} | "
    f"focus SIT/STAND(5,17) {focus_ss_acc:.4f} | "
    f"other SIT/STAND(11,23,29) {other_ss_acc:.4f} | "
    f"SIT<->STAND errors {val_ss_err} | dynamic errors {val_dynamic_err} | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | "
    f"params {model.count_params():,} | train sec {training_seconds:.1f} | "
    f"inference ms/sample {inference_ms_per_sample:.4f}"
)

## 17. 기록할 결과표

먼저 residual을 켠 B1을 seed 0·1·2로 실행합니다. 그다음 shortcut의 효과를 분리해서 보고 싶을 때만 `USE_RESIDUAL=False`로 동일 seed를 실행합니다.

| 실험 | seed | Val Accuracy | Macro F1 | Focus SIT/STAND | Params | Best epoch | Train sec | ms/sample |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| B1-residual | 0 |  |  |  |  |  |  |  |
| B1-residual | 1 |  |  |  |  |  |  |  |
| B1-residual | 2 |  |  |  |  |  |  |  |
| B1-no-skip | 0 |  |  |  |  |  |  |  |
| B1-no-skip | 1 |  |  |  |  |  |  |  |
| B1-no-skip | 2 |  |  |  |  |  |  |  |

### 해석 기준

- B1이 기존 CNN보다 좋으면 **결합 구조 전체가 유리했다**고 말할 수 있습니다.
- residual on/off가 같은 Inception 구조에서 안정적으로 차이 나면 shortcut의 효과를 더 직접적으로 말할 수 있습니다.
- 한 seed의 최고점이 아니라 세 seed의 평균과 표준편차로 판단합니다.
- validation 결과로 구조를 선택한 뒤에만 아래 공식 test 잠금을 해제합니다.

## 18. 🔒 공식 test 평가

모든 후보를 같은 validation 조건으로 비교하고 최종 모델을 결정한 뒤 한 번만 실행합니다. “모두 실행”에서 이 셀이 `AssertionError`로 멈추는 것은 의도한 동작입니다.

In [ ]:
FINAL_TEST = False

assert FINAL_TEST, (
    "공식 test는 최종 모델을 validation으로 선택한 뒤 한 번만 봅니다. "
    "최종 평가 시에만 FINAL_TEST=True로 변경하세요."
)

y_test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
cm_test = confusion_matrix(y_test, y_test_pred, labels=range(n_outputs))

print(f"test accuracy : {test_accuracy:.4f}")
print(f"test macro F1 : {test_f1:.4f}")
print(classification_report(
    y_test, y_test_pred, target_names=class_names,
    digits=4, zero_division=0,
))
print("confusion matrix")
print(cm_test)

test_ss_err = int(cm_test[SIT, STAND] + cm_test[STAND, SIT])
test_dynamic_err = int(cm_test[0:3, 0:3].sum() - np.trace(cm_test[0:3, 0:3]))

print(
    f"{EXPERIMENT_NAME} TEST | seed {SEED} | acc {test_accuracy:.4f} | "
    f"macro F1 {test_f1:.4f} | SIT<->STAND {test_ss_err} | "
    f"dynamic errors {test_dynamic_err}"
)

## 19. 참고 자료

- Szegedy et al., *Inception-v4, Inception-ResNet and the Impact of Residual Connections on Learning*, 2016
- He et al., *Deep Residual Learning for Image Recognition*, 2015
- Ismail Fawaz et al., *InceptionTime: Finding AlexNet for Time Series Classification*, 2019
- UCI Machine Learning Repository, *Human Activity Recognition Using Smartphones*